### Fraud Detection Project - Day 2
#### Objective
Build a reusable preprocessing pipeline for model training.

Tasks:
1. Load raw dataset
2. Drop unnecessary columns
3. Separate features and target
4. Encode categorical variables
5. Create train-test split
6. Save processed datasets

In [1]:
# =====================================
# IMPORTS & CONFIGURATION
# =====================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = "D:/Projects/fraud_detection_platform/data/raw/fraud_data.csv"
df = pd.read_csv(DATA_PATH,dtype={ "step": "int32","amount": "float32","isFraud": "int8","isFlaggedFraud": "int8"})
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
print("Columns:")
print(df.columns.tolist())

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


##### Drop High Cardinality ID Columns
nameOrig and nameDest are account identifiers.
Reasons:
- Millions of unique values
- Cause overfitting
- Poor generalization
- High memory cost

These may be used later for behavioral feature engineering.

In [4]:
df = df.drop(columns=["nameOrig","nameDest"])
print(df.shape)
df.head()

(6362620, 9)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.000000,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,41554.0,29885.86,0.0,0.0,0,0


In [5]:
## Separate Features and Targe
X = df.drop("isFraud", axis=1)
y = df["isFraud"]
print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)
#check datatypes
print(X.dtypes)

Feature Shape: (6362620, 8)
Target Shape: (6362620,)
step                int32
type               object
amount            float32
oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
isFlaggedFraud       int8
dtype: object


In [6]:
## Encode Transaction Type
##Because Machine learning models cannot directly use string values. Eg:-TRANSFER,PAYMENT,CASH_OUT
X = pd.get_dummies(X,columns=["type"],drop_first=False,dtype="int8")
print("Encoded Shape:", X.shape)
X.head()

Encoded Shape: (6362620, 12)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.639648,170136.0,160296.36,0.0,0.0,0,0,0,0,1,0
1,1,1864.280029,21249.0,19384.72,0.0,0.0,0,0,0,0,1,0
2,1,181.000000,181.0,0.00,0.0,0.0,0,0,0,0,0,1
3,1,181.000000,181.0,0.00,21182.0,0.0,0,0,1,0,0,0
4,1,11668.139648,41554.0,29885.86,0.0,0.0,0,0,0,0,1,0


In [7]:
#verify for no object columns present in dataset...
print(X.select_dtypes(include="object").columns.tolist())

[]


##### Train Test Split
Fraud dataset is highly imbalanced.
Use stratified sampling to preserve fraud ratio
in both train and test datasets.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
#checking shapes of data
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5090096, 12)
X_test : (1272524, 12)
y_train: (5090096,)
y_test : (1272524,)


In [9]:
#verify for fraud ratio
#We Aim ti make fraud ratio is equal in both train and test data...
print("Train Fraud %")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest Fraud %")
print(y_test.value_counts(normalize=True) * 100)

Train Fraud %
isFraud
0    99.870926
1     0.129074
Name: proportion, dtype: float64

Test Fraud %
isFraud
0    99.870887
1     0.129113
Name: proportion, dtype: float64


##### Save Processed Data
Store processed datasets for future training.

In [10]:
#CREATE OUTPUT PATHS
train_df = X_train.copy()
train_df["isFraud"] = y_train

test_df = X_test.copy()
test_df["isFraud"] = y_test

In [17]:
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print("Original Shape :", df.shape)
print("Train Shape    :", train_df.shape)
print("Test Shape     :", test_df.shape)

PREPROCESSING COMPLETE
Original Shape : (6362620, 9)
Train Shape    : (5090096, 13)
Test Shape     : (1272524, 13)
